# r–ρ–MI case study — channel-space explainer + run processing (2026-06-22)

Phase portraits of the channel–channel space $X^{(i)}$ vs $X^{(j)}$ per pair-type across SNR,
a $\beta$-sweep of the L–M signature, and processing of the `260622_g-roll` run.

**Construction.** Shared AR(1) mother $z$ (standardised). Each channel = filter($z$),
standardised to unit variance, plus i.i.d. Gaussian noise. Because the signal is unit-variance,
`noise_std` alone sets SNR ($=1/\text{noise\_std}^2$); `ar1_a` controls mother smoothness, not strength.

Filters: **L** $= z$ | **M** $=$ sigmoid($\beta z$) | **NM** $= (z-\bar z)^2$ (symmetric $\Rightarrow r=\rho=0$).

The generator is `src.generators.generate_filter_roll_mts`; the run is driven by
`configs/generate/r_rho_mi/260622_g-roll.yaml` (3 iteration classes).

In [ ]:
import numpy as np
from scipy.stats import spearmanr
from sklearn.feature_selection import mutual_info_regression
import matplotlib.pyplot as plt

def ar1_mother(T, a=0.8, rng=None):
    rng = rng or np.random.default_rng(0)
    m = np.zeros(T)
    for t in range(1, T):
        m[t] = a * m[t-1] + rng.normal(0, 1)
    return (m - m.mean()) / m.std()

def unit(g):
    return (g - g.mean()) / g.std()

def sigmoid(z, beta):           # monotone-nonlinear filter, gain beta
    return 1.0 / (1.0 + np.exp(-beta * z))

def mi(a, b):
    return float(mutual_info_regression(a.reshape(-1, 1), b, n_neighbors=4, random_state=0)[0])

T = 2000
z = ar1_mother(T, a=0.8, rng=np.random.default_rng(4))
beta = 2.5                      # <-- tweak me
L  = unit(z)
M  = unit(sigmoid(z, beta))
NM = unit((z - z.mean())**2)

## Channel–channel spaces: pair-type × SNR

In [ ]:
pairs = {"L-L": (L, L), "M-M": (M, M), "NM-NM": (NM, NM),
         "L-M": (L, M), "L-NM": (L, NM), "M-NM": (M, NM)}
noise = [0.1, 0.3, 0.6, 1.0, 1.6]          # noise_std on unit-variance signal; SNR = 1/std^2

rng = np.random.default_rng(1)
nr, nc = len(pairs), len(noise)
fig, axes = plt.subplots(nr, nc, figsize=(2.5*nc, 2.5*nr))
for i, (name, (a0, b0)) in enumerate(pairs.items()):
    for j, ns in enumerate(noise):
        xi, xj = a0 + rng.normal(0, ns, T), b0 + rng.normal(0, ns, T)
        rr, rho, mm = np.corrcoef(xi, xj)[0, 1], spearmanr(xi, xj)[0], mi(xi, xj)
        ax = axes[i, j]
        ax.scatter(xi, xj, s=2, alpha=0.15, color="navy", linewidths=0)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title((f"SNR={1/ns**2:.0f}\n" if i == 0 else "") + f"r={rr:+.2f} rho={rho:+.2f} MI={mm:.2f}", fontsize=7)
        if j == 0:
            ax.set_ylabel(name, fontweight="bold", fontsize=11)
fig.suptitle(f"Channel-channel space (sigmoid beta={beta}): pair-type (rows) x SNR (cols)", y=0.995)
fig.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()

## β-sweep (single L–M pair): how far does monotone nonlinearity break {r, ρ}?

In [ ]:
# Can monotone nonlinearity break {r, rho}?  Sweep sigmoid gain beta for an L-M pair.
betas = [0.5, 1, 2, 3, 4, 6, 8, 12, 20]
noise_lo = 0.05
rng = np.random.default_rng(2)
rows = []
for b in betas:
    Mb = unit(sigmoid(z, b))
    xi, xj = unit(z) + rng.normal(0, noise_lo, T), Mb + rng.normal(0, noise_lo, T)
    rows.append((b, np.corrcoef(xi, xj)[0, 1], spearmanr(xi, xj)[0], mi(xi, xj)))
rows = np.array(rows)

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(rows[:, 0], rows[:, 1], "o-", label="r (Pearson)")
ax.plot(rows[:, 0], rows[:, 2], "s-", label="rho (Spearman)")
ax.axhline(np.sqrt(2/np.pi), ls="--", color="grey", lw=0.8, label="sqrt(2/pi) = corr(z, sign z)")
ax.set_xlabel("sigmoid gain beta"); ax.set_ylabel("correlation"); ax.set_xscale("log")
ax.set_title("L-M pair: rho ~1 then falls as beta->step (ties); r floors ~0.80.\nMax r-rho gap at moderate beta.")
ax.legend(); fig.tight_layout(); plt.show()
print("beta, r, rho, MI, (rho-r) gap:")
for b, r_, rho_, mm_ in rows:
    print(f"  beta={b:5.1f}  r={r_:.3f}  rho={rho_:.3f}  MI={mm_:.3f}  gap={rho_-r_:+.3f}")

## MTS-level β: which β drops the *feature* corr(r, ρ)?

The per-pair (ρ−r) gap above peaks at *moderate* β, but what we tune for is the MTS-level
**feature** `corr(r, ρ)` over all pairs. Real pyspi (M=20, T=2000, iter-2 L+M, 3 instances):
`corr(r, ρ)` falls **monotonically** with β — π→0.88, 4→0.83, 2π→0.78 — while `corr(ρ, MI)`
stays ~0.92 at *every* β. So β cannot contaminate the iter-2 message; higher β just deepens the
drop while the sigmoid becomes more step-like. **The config uses β=4.** (Cell below uses an
sklearn MI proxy so it runs without pyspi; the trend matches.)

In [ ]:
from src.generators import generate_filter_roll_mts

def _spi_corrs(X):
    R = np.corrcoef(X, rowvar=False)
    S = spearmanr(X)[0]
    Mn = X.shape[1]; I = np.zeros((Mn, Mn))
    for i in range(Mn):
        for j in range(i + 1, Mn):
            I[i, j] = I[j, i] = mutual_info_regression(X[:, [i]], X[:, j], n_neighbors=4, random_state=0)[0]
    iu = np.triu_indices(Mn, 1)
    fc = lambda a, b: np.corrcoef(a, b)[0, 1]
    return fc(R[iu], S[iu]), fc(S[iu], I[iu])

betas = [1, 2, np.pi, 4, 2*np.pi, 8]
rows = []
for b in betas:
    X = generate_filter_roll_mts(M=20, T=2000, n_linear=10, n_monotonic=10, beta=b,
                                 noise_std=0.1, noise_std_variable=True, noise_std_scale=2.0,
                                 ar1_a=0.8, zscore=True, rng=np.random.default_rng(100))
    rows.append((b, *_spi_corrs(X)))
rows = np.array(rows)

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(rows[:, 0], rows[:, 1], "o-", label="corr(r, rho)  <- want this to drop")
ax.plot(rows[:, 0], rows[:, 2], "s-", label="corr(rho, MI)  <- want this high")
for x in (np.pi, 2*np.pi):
    ax.axvline(x, ls=":", color="grey", lw=0.8)
ax.set_xscale("log"); ax.set_xlabel("sigmoid gain beta"); ax.set_ylabel("meta-feature corr")
ax.set_title("iter-2 (L+M): corr(r,rho) drops with beta; corr(rho,MI) stays high")
ax.legend(); fig.tight_layout(); plt.show()
print(rows)

## Process the `260622_g-roll` run: staggered meta-feature table

Loads `data/r_rho_mi/260622_g-roll/<iter>/M20_T2000_I*/spi_mpis.npz`, computes the three
`corr(SPI_i, SPI_j)` meta-features per instance (Pearson over MPI off-diagonals), and aggregates
over the 10 instances. **Expect:** `corr(r, ρ)` drops at iter2 (monotone nonlinearity); `corr(·, MI)`
drops at iter3 (non-monotonicity); `corr(r, ρ)` recovers at iter3 (r and ρ fail together).

In [ ]:
import json
from src.utils import project_root

BASE = project_root() / "data/r_rho_mi/260622_g-roll"
ITERS = ["iter1_L", "iter2_LM", "iter3_LMNM"]
FEATS = ["corr(r,rho)", "corr(r,MI)", "corr(rho,MI)"]

def _keymap(keys):
    out = {}
    for k in keys:
        kl = k.lower()
        if "cov" in kl or "empirical" in kl: out["r"] = k
        elif "spearman" in kl: out["rho"] = k
        elif "mi_" in kl or "kraskov" in kl or "mutual" in kl: out["MI"] = k
    return out

def _offdiag(Mm):
    iu = np.triu_indices(Mm.shape[0], 1); return Mm[iu]

def load_mpis(d):
    npz = np.load(d / "spi_mpis.npz"); km = _keymap(npz.files)
    return {k: npz[km[k]] for k in ("r", "rho", "MI")}

def meta_feats(mats):
    rv, sv, iv = _offdiag(mats["r"]), _offdiag(mats["rho"]), _offdiag(mats["MI"])
    fc = lambda a, b: float(np.corrcoef(a, b)[0, 1])
    return {"corr(r,rho)": fc(rv, sv), "corr(r,MI)": fc(rv, iv), "corr(rho,MI)": fc(sv, iv)}

means, stds = {p: [] for p in FEATS}, {p: [] for p in FEATS}
for it in ITERS:
    feats = [meta_feats(load_mpis(d)) for d in sorted((BASE / it).glob("M*_T*_I*"))
             if (d / "spi_mpis.npz").exists()]
    for p in FEATS:
        v = np.array([f[p] for f in feats])
        means[p].append(v.mean()); stds[p].append(v.std())

x = np.arange(len(ITERS))
fig, ax = plt.subplots(figsize=(7, 4.5))
for p in FEATS:
    ax.errorbar(x, means[p], yerr=stds[p], marker="o", capsize=4, label=p)
ax.set_xticks(x); ax.set_xticklabels(ITERS)
ax.set_ylabel("meta-feature  corr(SPI_i, SPI_j)")
ax.set_title("Staggered decoupling: corr(r,rho) drops at iter2, corr(.,MI) drops at iter3")
ax.legend(); ax.grid(True, alpha=0.3); fig.tight_layout(); plt.show()

print(f"{'iter':>12} " + " ".join(f"{p:>16}" for p in FEATS))
for i, it in enumerate(ITERS):
    print(f"{it:>12} " + " ".join(f"{means[p][i]:7.3f}+/-{stds[p][i]:.3f}" for p in FEATS))

## Per-pair-type SPI–SPI scatter (iter3)

Each point is one channel pair in the MPI, coloured by its pair-type (from `meta.json` `types`).
This shows *which* pairs move where: **L–NM / M–NM** pairs collapse to `r=ρ≈0` with high MI — that
is what drags `corr(·, MI)` down while `corr(r, ρ)` stays high (r and ρ fail together).

In [ ]:
d = BASE / "iter3_LMNM" / "M20_T2000_I0"
types = json.loads((d / "meta.json").read_text())["generator"]["types"]
mats = load_mpis(d)
abbr = {"linear": "L", "monotonic": "M", "non-monotonic": "NM"}
Mn = len(types)
pt = {}
for i in range(Mn):
    for j in range(i + 1, Mn):
        t = "-".join(sorted([abbr[types[i]], abbr[types[j]]]))
        pt.setdefault(t, []).append((mats["r"][i, j], mats["rho"][i, j], mats["MI"][i, j]))

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for t, vals in pt.items():
    v = np.array(vals)
    axes[0].scatter(v[:, 0], v[:, 1], s=30, alpha=0.7, label=t)
    axes[1].scatter(v[:, 1], v[:, 2], s=30, alpha=0.7, label=t)
axes[0].set_xlabel("r (Pearson)"); axes[0].set_ylabel("rho (Spearman)"); axes[0].set_title("(r, rho) plane")
axes[1].set_xlabel("rho (Spearman)"); axes[1].set_ylabel("MI (Kraskov)"); axes[1].set_title("(rho, MI) plane")
for ax in axes:
    ax.grid(True, alpha=0.3); ax.legend(title="pair-type", fontsize=8)
fig.suptitle("iter3 MPI pairs by channel-pair type: NM pairs (r,rho~0, MI high) break corr(.,MI)")
fig.tight_layout(); plt.show()